# **MODULE 8 — LANGSMITH MASTERCLASS**

## **The 4 Important Configuration Values**

1. **LANGSMITH_API_KEY** — Connects the application to LangSmith.
2. **LANGSMITH_TRACING** — Enables LangSmith tracing.
3. **LANGSMITH_PROJECT** — Defines the LangSmith project where traces are stored.
4. **OPENAI_API_KEY** — Allows the application to use the OpenAI model.

In [1]:
import os

# LangSmith + OpenAI configuration
os.environ["LANGSMITH_API_KEY"] = "your langsmith key"          # from smith.langchain.com
os.environ["LANGSMITH_TRACING"] = "true"                                                          # turn tracing ON
os.environ["LANGSMITH_PROJECT"] = "PROITBRIDGE_04"
os.environ["OPENAI_API_KEY"] = "your openAI api key here" # from platform.openai.com

print("Configuration set.")

Configuration set.


## **INSTALL REQUIRED PACKAGES**

In [3]:
!pip install -q -U "langchain>=1.0,<2.0" langsmith langchain-openai langgraph langchain-community pypdf

## **Imports**

In [4]:
from langsmith import traceable
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

print("Imports ready.")

/tmp/ipykernel_37909/832546325.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports ready.


#==============================================================================

# **PART 1 — SIMPLE PYTHON FUNCTION TRACE**

## **First LangSmith Trace**

**`@traceable` tells LangSmith to record the execution of this function.**

In [5]:
@traceable
def calculate_salary(basic_salary, bonus):
    # simple HR calculation
    return basic_salary + bonus

result = calculate_salary(50000, 10000)
print("Total salary:", result)

Total salary: 60000


#==============================================================================

# **PART 2 — RAG TRACING**

In [6]:
loader = PyPDFLoader("PROITBRIDGE_Employee_Handbook_2026_1.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)

print("Handbook pages:", len(pages), "| chunks:", len(chunks))

Handbook pages: 57 | chunks: 142


In [7]:
embeddings   = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever    = vector_store.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

@tool
def search_handbook(query: str) -> str:
    """Search the PROITBRIDGE Employee Handbook and return relevant handbook content."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant handbook content was found."
    return "\n\n".join(doc.page_content for doc in docs)

rag_agent = create_agent(model=llm,
                         tools=[search_handbook],
                         system_prompt=("You are an HR handbook assistant. Answer using only information "
                                        "returned by the search_handbook tool. If the handbook does not "
                                        "contain the answer, say that the information was not found."))
print("RAG agent ready.")

RAG agent ready.


## **Generate the RAG Trace**

In [8]:
result = rag_agent.invoke({"messages": [{"role": "user", "content": "How many casual leave (CL) days do employees get per year?"}]})
print("Answer  :", result["messages"][-1].content)

Answer  : Employees get 8 casual leave (CL) days per year. These 8 days are credited upfront on January 1 each year. Casual leave is intended for short, unplanned personal needs and has a maximum of 3 consecutive days at a time. It cannot be combined with earned leave (EL). Casual leave does not accrue or carry forward and lapses if not used within the year.


In [10]:
question = "Can I take 4 days of casual leave in a row?"
result = rag_agent.invoke(
                          {"messages": [{"role": "user", "content": question}]},
                          config={"run_name": "RAG"}
                          )

print("Question:", question)
print("Answer  :", result["messages"][-1].content)

Question: Can I take 4 days of casual leave in a row?
Answer  : You cannot take 4 days of casual leave in a row. The maximum allowed consecutive casual leave is 3 days at a time.


#==============================================================================

# **THE END.**